### Synthetic Dataset Creation

In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np

In [ ]:

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import uuid

# =====================================================
# CONFIGURATION
# =====================================================

np.random.seed(42)
random.seed(42)

N_USERS = 50000

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2025, 12, 31)


# COMPANIES


companies = [
    ("C001", "TechNova", "Electronics"),
    ("C002", "StyleHub", "Fashion"),
    ("C003", "FitLife", "Fitness"),
    ("C004", "HomeSphere", "Home Decor"),
    ("C005", "FoodDash", "Food Delivery"),
    ("C006", "BookVerse", "E-Commerce"),
    ("C007", "TravelEase", "Travel"),
    ("C008", "FinSecure", "FinTech")
]

# CAMPAIGNS


campaigns = []

for company_id, company_name, industry in companies:
    campaigns.extend([
        (
            f"{company_id}_CMP1",
            f"{company_name}_Summer",
            company_id,
            company_name,
            industry
        ),
        (
            f"{company_id}_CMP2",
            f"{company_name}_Festive",
            company_id,
            company_name,
            industry
        ),
        (
            f"{company_id}_CMP3",
            f"{company_name}_Clearance",
            company_id,
            company_name,
            industry
        )
    ])


# MASTER LISTS


platforms = [
    "Google Ads",
    "Facebook Ads",
    "Instagram Ads",
    "YouTube Ads"
]

cities = [
    "Mumbai",
    "Delhi",
    "Bangalore",
    "Pune",
    "Hyderabad",
    "Chennai",
    "Kolkata",
    "Ahmedabad"
]

device_types = [
    "Mobile",
    "Desktop",
    "Tablet"
]

age_groups = [
    "18-24",
    "25-34",
    "35-44",
    "45+"
]

traffic_sources = [
    "Paid Search",
    "Social",
    "Referral",
    "Organic"
]

customer_types = [
    "New",
    "Returning"
]

variants = [
    "A",
    "B"
]


# FUNNEL PROBABILITIES


CTR = {
    "Google Ads": 0.12,
    "Facebook Ads": 0.09,
    "Instagram Ads": 0.08,
    "YouTube Ads": 0.06
}

LANDING_RATE = 0.90

PRODUCT_VIEW_RATE = 0.80

ADD_TO_CART_RATE = 0.40

PURCHASE_RATE = {
    "Electronics": 0.45,
    "Fashion": 0.60,
    "Fitness": 0.55,
    "Home Decor": 0.50,
    "Food Delivery": 0.75,
    "E-Commerce": 0.65,
    "Travel": 0.35,
    "FinTech": 0.25
}
campaign_ab_config = {}

for campaign in campaigns:

    campaign_name = campaign[1]

    campaign_ab_config[campaign_name] = {

        "A_effectiveness": round(
            random.uniform(0.95, 1.15),
            3
        ),

        "B_effectiveness": round(
            random.uniform(0.95, 1.15),
            3
        )
    }

# STORAGE


records = []


# GENERATION LOOP


for user_num in range(1, N_USERS + 1):

    user_id = f"U{user_num:06d}"

    session_id = str(uuid.uuid4())

    campaign = random.choice(campaigns)

    campaign_id = campaign[0]
    campaign_name = campaign[1]
    company_id = campaign[2]
    company_name = campaign[3]
    industry = campaign[4]

    platform = random.choice(platforms)

    city = random.choice(cities)

    device = random.choices(
        ["Mobile", "Desktop", "Tablet"],
        weights=[65, 25, 10]
    )[0]

    age_group = random.choice(age_groups)

    traffic_source = random.choice(traffic_sources)

    customer_type = random.choices(
        customer_types,
        weights=[70, 30]
    )[0]

    variant = random.choice(["A", "B"])

    random_days = random.randint(
        0,
        (END_DATE - START_DATE).days
    )

    base_date = START_DATE + timedelta(
        days=random_days,
        hours=random.randint(8, 22),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )

    # SEASONALITY


    seasonal_multiplier = 1.0

    if base_date.month in [10, 11, 12]:
        seasonal_multiplier = 1.30

    # DEVICE EFFECT


    device_multiplier = {
        "Desktop": 1.20,
        "Mobile": 1.00,
        "Tablet": 0.90
    }[device]


    # CAMPAIGN LEVEL A/B TEST


    if variant == "A":

        variant_multiplier = (
            campaign_ab_config[campaign_name]
            ["A_effectiveness"]
        )

    else:

        variant_multiplier = (
            campaign_ab_config[campaign_name]
            ["B_effectiveness"]
        )

    purchase_multiplier = (
        seasonal_multiplier *
        device_multiplier *
        variant_multiplier
    )

    # IMPRESSION


    records.append([
        user_id,
        session_id,
        base_date,
        company_id,
        company_name,
        industry,
        campaign_id,
        campaign_name,
        platform,
        variant,
        city,
        device,
        age_group,
        traffic_source,
        customer_type,
        "impression",
        0
    ])


    # CLICK

    if np.random.rand() < CTR[platform]:

        click_time = base_date + timedelta(
            minutes=np.random.randint(1, 30)
        )

        records.append([
            user_id,
            session_id,
            click_time,
            company_id,
            company_name,
            industry,
            campaign_id,
            campaign_name,
            platform,
            variant,
            city,
            device,
            age_group,
            traffic_source,
            customer_type,
            "click",
            0
        ])

      
        # LANDING PAGE
      

        if np.random.rand() < LANDING_RATE:

            landing_time = click_time + timedelta(
                minutes=np.random.randint(1, 10)
            )

            records.append([
                user_id,
                session_id,
                landing_time,
                company_id,
                company_name,
                industry,
                campaign_id,
                campaign_name,
                platform,
                variant,
                city,
                device,
                age_group,
                traffic_source,
                customer_type,
                "landing_page",
                0
            ])

            # PRODUCT VIEW
         

            if np.random.rand() < PRODUCT_VIEW_RATE:

                product_time = landing_time + timedelta(
                    minutes=np.random.randint(1, 15)
                )

                records.append([
                    user_id,
                    session_id,
                    product_time,
                    company_id,
                    company_name,
                    industry,
                    campaign_id,
                    campaign_name,
                    platform,
                    variant,
                    city,
                    device,
                    age_group,
                    traffic_source,
                    customer_type,
                    "product_view",
                    0
                ])

                # ADD TO CART
       

                if np.random.rand() < ADD_TO_CART_RATE:

                    cart_time = product_time + timedelta(
                        minutes=np.random.randint(1, 20)
                    )

                    records.append([
                        user_id,
                        session_id,
                        cart_time,
                        company_id,
                        company_name,
                        industry,
                        campaign_id,
                        campaign_name,
                        platform,
                        variant,
                        city,
                        device,
                        age_group,
                        traffic_source,
                        customer_type,
                        "add_to_cart",
                        0
                    ])

                   
                    # PURCHASE
              

                    base_purchase_rate = PURCHASE_RATE[
                        industry
                    ]

                    final_purchase_rate = min(
                        base_purchase_rate *
                        purchase_multiplier,
                        0.95
                    )

                    if np.random.rand() < final_purchase_rate:

                        purchase_time = cart_time + timedelta(
                            days=np.random.randint(0, 7),
                            hours=np.random.randint(1, 24)
                        )

                        revenue = round(
                            max(
                                500,
                                np.random.normal(
                                    3000,
                                    1000
                                )
                            ),
                            2
                        )

                        records.append([
                            user_id,
                            session_id,
                            purchase_time,
                            company_id,
                            company_name,
                            industry,
                            campaign_id,
                            campaign_name,
                            platform,
                            variant,
                            city,
                            device,
                            age_group,
                            traffic_source,
                            customer_type,
                            "purchase",
                            revenue
                        ])

In [ ]:


import random
import uuid
from datetime import datetime, timedelta

import numpy as np
import pandas as pd


# REPRODUCIBILITY

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# CONFIGURATION

N_USERS = 50_000
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2025, 12, 31)
DATE_RANGE_DAYS = (END_DATE - START_DATE).days
CURRENT_BATCH = "BATCH_001"


# MASTER DATA
----
COMPANIES = [
    ("C001", "TechNova", "Electronics"),
    ("C002", "StyleHub", "Fashion"),
    ("C003", "FitLife", "Fitness"),
    ("C004", "HomeSphere", "Home Decor"),
    ("C005", "FoodDash", "Food Delivery"),
    ("C006", "BookVerse", "E-Commerce"),
    ("C007", "TravelEase", "Travel"),
    ("C008", "FinSecure", "FinTech"),
]

CAMPAIGN_TYPES = ["Summer", "Festive", "Clearance"]

CAMPAIGNS = [
    (f"{cid}_CMP{i+1}", f"{cname}_{ctype}", cid, cname, industry)
    for cid, cname, industry in COMPANIES
    for i, ctype in enumerate(CAMPAIGN_TYPES)
]

PLATFORMS = ["Google Ads", "Facebook Ads", "Instagram Ads", "YouTube Ads"]
CITIES = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Pune", "Chennai", "Ahmedabad", "Kolkata"]
DEVICE_TYPES = ["Mobile", "Desktop", "Tablet"]
AGE_GROUPS = ["18-24", "25-34", "35-44", "45+"]
TRAFFIC_SOURCES = ["Paid Search", "Social", "Organic", "Referral"]
CUSTOMER_TYPES = ["New", "Returning"]
VARIANTS = ["A", "B"]

CTR = {"Google Ads": 0.12, "Facebook Ads": 0.09, "Instagram Ads": 0.08, "YouTube Ads": 0.06}

LANDING_RATE = 0.90
PRODUCT_VIEW_RATE = 0.80
ADD_TO_CART_RATE = 0.40

PURCHASE_RATE = {
    "Electronics": 0.45, "Fashion": 0.60, "Fitness": 0.55, "Home Decor": 0.50,
    "Food Delivery": 0.75, "E-Commerce": 0.65, "Travel": 0.35, "FinTech": 0.25,
}

DEVICE_MULTIPLIER = {"Desktop": 1.15, "Mobile": 1.00, "Tablet": 0.90}
CUSTOMER_MULTIPLIER = {"New": 1.00, "Returning": 1.15}
TRAFFIC_MULTIPLIER = {"Paid Search": 1.15, "Organic": 1.10, "Referral": 1.05, "Social": 1.00}

REVENUE_RANGE = {
    "Electronics": (20000, 80000), "Fashion": (1000, 6000), "Fitness": (2000, 12000),
    "Home Decor": (5000, 25000), "Food Delivery": (300, 2000), "E-Commerce": (500, 8000),
    "Travel": (15000, 120000), "FinTech": (5000, 50000),
}

CAMPAIGN_AB_CONFIG = {
    campaign_name: {
        "A_effectiveness": round(random.uniform(0.95, 1.15), 3),
        "B_effectiveness": round(random.uniform(0.95, 1.15), 3),
    }
    for _, campaign_name, _, _, _ in CAMPAIGNS
}

COLUMNS = [
    "batch_id", "user_id", "session_id", "session_number", "event_timestamp",
    "company_id", "company_name", "industry", "campaign_id", "campaign_name",
    "platform", "ad_variant", "city", "device_type", "age_group", "traffic_source",
    "user_lifecycle", "order_id", "event_type", "revenue", "recovery_flag",
]



# HELPER FUNCTIONS

def generate_timestamp():
    dt = START_DATE + timedelta(
        days=random.randint(0, DATE_RANGE_DAYS),
        hours=random.randint(8, 22),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59),
    )
    return dt


def seasonal_multiplier(timestamp):
    return 1.30 if timestamp.month in (10, 11, 12) else 1.00


def generate_order_id():
    return "ORD-" + str(uuid.uuid4())[:8]


def generate_revenue(industry):
    low, high = REVENUE_RANGE[industry]
    return round(random.uniform(low, high), 2)


# USER & SESSION GENERATION

records = []

for user_num in range(1, N_USERS + 1):
    user_id = f"USR_{user_num:06d}"

    city = random.choice(CITIES)
    device = random.choices(DEVICE_TYPES, weights=[65, 25, 10])[0]
    age_group = random.choice(AGE_GROUPS)
    traffic_source = random.choice(TRAFFIC_SOURCES)
    customer_type = random.choices(CUSTOMER_TYPES, weights=[70, 30])[0]

    lifecycle = (
        "New" if customer_type == "New"
        else random.choices(["Returning", "Loyal"], weights=[75, 25])[0]
    )

    n_sessions = np.random.choice([1, 2, 3, 4, 5], p=[0.35, 0.30, 0.20, 0.10, 0.05])

    for session_number in range(1, n_sessions + 1):
        session_id = str(uuid.uuid4())

        campaign_id, campaign_name, company_id, company_name, industry = random.choice(CAMPAIGNS)
        platform = random.choice(PLATFORMS)
        variant = random.choice(VARIANTS)
        variant_effectiveness = CAMPAIGN_AB_CONFIG[campaign_name][f"{variant}_effectiveness"]

        session_start = generate_timestamp()

        combined_mult = (
            seasonal_multiplier(session_start)
            * DEVICE_MULTIPLIER[device]
            * CUSTOMER_MULTIPLIER[customer_type]
            * TRAFFIC_MULTIPLIER[traffic_source]
            * variant_effectiveness
        )

        click_probability = min(CTR[platform] * combined_mult, 0.95)
        purchase_probability = min(PURCHASE_RATE[industry] * combined_mult, 0.95)

        # Common fields shared by every event row in this session
        base = [
            CURRENT_BATCH, user_id, session_id, session_number,
            company_id, company_name, industry, campaign_id, campaign_name,
            platform, variant, city, device, age_group, traffic_source, lifecycle,
        ]

        def emit(ts, event_type, order_id=None, revenue=0, recovery=False):
            records.append([*base[:4], ts, *base[4:], order_id, event_type, revenue, recovery])

        # ---------------- FUNNEL ----------------
        emit(session_start, "impression")

        if np.random.rand() < click_probability:
            click_time = session_start + timedelta(minutes=random.randint(1, 5))
            emit(click_time, "click")

            if np.random.rand() < LANDING_RATE:
                landing_time = click_time + timedelta(seconds=random.randint(10, 120))
                emit(landing_time, "landing_page")

                if np.random.rand() < PRODUCT_VIEW_RATE:
                    product_time = landing_time + timedelta(minutes=random.randint(1, 8))
                    emit(product_time, "product_view")

                    if np.random.rand() < ADD_TO_CART_RATE:
                        cart_time = product_time + timedelta(minutes=random.randint(1, 10))
                        emit(cart_time, "add_to_cart")

                        if np.random.rand() < purchase_probability:
                            purchase_time = cart_time + timedelta(
                                days=random.randint(0, 5),
                                hours=random.randint(0, 12),
                                minutes=random.randint(0, 59),
                            )
                            emit(
                                purchase_time, "purchase",
                                order_id=generate_order_id(),
                                revenue=generate_revenue(industry),
                            )
                        else:
                            recovery_probability = 0.50 if customer_type == "Returning" else 0.35
                            if np.random.rand() < recovery_probability:
                                recovery_time = cart_time + timedelta(
                                    days=random.randint(1, 7),
                                    hours=random.randint(1, 20),
                                )
                                emit(
                                    recovery_time, "purchase",
                                    order_id=generate_order_id(),
                                    revenue=generate_revenue(industry),
                                    recovery=True,
                                )


# DATAFRAME

df = pd.DataFrame(records, columns=COLUMNS)
df = df.sort_values(by=["event_timestamp", "user_id", "session_number"]).reset_index(drop=True)


# DATA VALIDATION

print("\nChecking Missing Values...\n", df.isnull().sum())
print("\nChecking Duplicate Rows...\n", df.duplicated().sum())
print("\nEvent Distribution\n", df["event_type"].value_counts())
print("\nPlatform Distribution\n", df["platform"].value_counts())
print("\nCampaign Distribution\n", df["campaign_name"].value_counts().head(10))
print("\nIndustry Distribution\n", df["industry"].value_counts())
print("\nLifecycle Distribution\n", df["user_lifecycle"].value_counts())
print("\nRecovery Purchases\n", df["recovery_flag"].value_counts())

# SUMMARY METRICS

total_users = df["user_id"].nunique()
total_sessions = df["session_id"].nunique()
purchases = df[df["event_type"] == "purchase"]
total_orders = purchases["order_id"].nunique()
total_revenue = df["revenue"].sum()
average_order_value = purchases["revenue"].mean()
conversion_rate = total_orders / total_users * 100

print("\n" + "=" * 70)
print("USER JOURNEY DATASET GENERATED SUCCESSFULLY")
print("=" * 70)
print(f"Batch ID                : {CURRENT_BATCH}")
print(f"Total Users             : {total_users:,}")
print(f"Total Sessions          : {total_sessions:,}")
print(f"Total Orders            : {total_orders:,}")
print(f"Total Revenue           : \u20b9{total_revenue:,.2f}")
print(f"Average Order Value     : \u20b9{average_order_value:,.2f}")
print(f"Conversion Rate         : {conversion_rate:.2f}%")
print(f"Total Events            : {len(df):,}")
print("=" * 70)


# EXPORT

OUTPUT_FILE = "user_journey_events.csv"
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nDataset exported successfully to {OUTPUT_FILE}")

print("\nFirst 10 Records\n")
print(df.head(10))



Checking Missing Values...
 batch_id                0
user_id                 0
session_id              0
session_number          0
event_timestamp         0
company_id              0
company_name            0
industry                0
campaign_id             0
campaign_name           0
platform                0
ad_variant              0
city                    0
device_type             0
age_group               0
traffic_source          0
user_lifecycle          0
order_id           146370
event_type              0
revenue                 0
recovery_flag           0
dtype: int64

Checking Duplicate Rows...
 0

Event Distribution
 event_type
impression      110169
click            12516
landing_page     11196
product_view      8902
add_to_cart       3587
purchase          2871
Name: count, dtype: int64

Platform Distribution
 platform
Google Ads       41269
Facebook Ads     37554
Instagram Ads    36148
YouTube Ads      34270
Name: count, dtype: int64

Campaign Distribution
 campaign_n